# Leaf Anomaly Detection — Model Training

This notebook trains the `LeafAnomalyModel` (ResNet-50 + CBAM + MAML)
on the PlantVillage dataset (or any similarly structured leaf disease dataset).

In [ ]:
import sys
sys.path.append('../backend')

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision.transforms as T
import torchvision.datasets as datasets
import matplotlib.pyplot as plt
from pathlib import Path

from app.models.leaf_model import LeafAnomalyModel
from app.config import IMAGENET_MEAN, IMAGENET_STD, NUM_CLASSES, SAVED_MODELS_DIR

DEVICE     = 'cuda' if torch.cuda.is_available() else 'cpu'
BATCH_SIZE = 32
EPOCHS     = 30
LR         = 1e-4
print(f'Using device: {DEVICE}')

## 1 — Data Augmentation & Loading

In [ ]:
train_transforms = T.Compose([
    T.RandomResizedCrop(224, scale=(0.7, 1.0)),
    T.RandomHorizontalFlip(),
    T.RandomVerticalFlip(),
    T.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2, hue=0.1),
    T.RandomRotation(30),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

val_transforms = T.Compose([
    T.Resize(256),
    T.CenterCrop(224),
    T.ToTensor(),
    T.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

DATA_ROOT = Path('../backend/dataset')
train_ds  = datasets.ImageFolder(DATA_ROOT / 'train',      transform=train_transforms)
val_ds    = datasets.ImageFolder(DATA_ROOT / 'validation', transform=val_transforms)
test_ds   = datasets.ImageFolder(DATA_ROOT / 'test',       transform=val_transforms)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,  num_workers=4)
val_loader   = DataLoader(val_ds,   batch_size=BATCH_SIZE, shuffle=False, num_workers=4)
test_loader  = DataLoader(test_ds,  batch_size=BATCH_SIZE, shuffle=False, num_workers=4)

print(f'Train: {len(train_ds)} | Val: {len(val_ds)} | Test: {len(test_ds)}')
print(f'Classes: {train_ds.classes}')

## 2 — Model Initialization

In [ ]:
model     = LeafAnomalyModel(num_classes=NUM_CLASSES, pretrained=True).to(DEVICE)
criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)

total_params     = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

## 3 — Training Loop

In [ ]:
def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

@torch.no_grad()
def eval_epoch(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)
        total_loss += loss.item() * imgs.size(0)
        correct    += (logits.argmax(1) == labels).sum().item()
        total      += imgs.size(0)
    return total_loss / total, correct / total

history    = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
best_acc   = 0.0

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc = train_epoch(model, train_loader, optimizer, criterion, DEVICE)
    vl_loss, vl_acc = eval_epoch(model,  val_loader,   criterion, DEVICE)
    scheduler.step()

    history['train_loss'].append(tr_loss); history['train_acc'].append(tr_acc)
    history['val_loss'].append(vl_loss);   history['val_acc'].append(vl_acc)

    print(f'Epoch {epoch:02d}/{EPOCHS}  '
          f'Train Loss: {tr_loss:.4f} Acc: {tr_acc:.4f}  '
          f'Val Loss: {vl_loss:.4f} Acc: {vl_acc:.4f}')

    if vl_acc > best_acc:
        best_acc = vl_acc
        SAVED_MODELS_DIR.mkdir(exist_ok=True)
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch, 'best_acc': best_acc},
                   SAVED_MODELS_DIR / 'leaf_detection_model.pth')
        print(f'  ✓ Saved best model (val acc: {best_acc:.4f})')

## 4 — Training Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].plot(history['train_loss'], label='Train Loss', color='#22c55e')
axes[0].plot(history['val_loss'],   label='Val Loss',   color='#ef4444')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(history['train_acc'], label='Train Acc', color='#22c55e')
axes[1].plot(history['val_acc'],   label='Val Acc',   color='#ef4444')
axes[1].set_title('Accuracy'); axes[1].legend(); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5 — Test Evaluation

In [ ]:
checkpoint = torch.load(SAVED_MODELS_DIR / 'leaf_detection_model.pth', map_location=DEVICE)
model.load_state_dict(checkpoint['model_state_dict'])
test_loss, test_acc = eval_epoch(model, test_loader, criterion, DEVICE)
print(f'Test Loss: {test_loss:.4f}  Test Accuracy: {test_acc:.4f} ({test_acc*100:.2f}%)')